In [ ]:
# Install Hugging Face and supporting libraries for NLP and datasets, and OpenAI Whisper

!pip install "transformers" "datasets" "accelerate" "evaluate" "sacremoses" "pyarrow"
!pip install -q openai-whisper



In [ ]:

# import pandas, PyTorch, Dataset, DataLoader, transformers,AutoTokenizer, AutoModelForCausalLM,
#Name of the pretrained model,
# Download and load the tokenizer for BioGPT, Download and load the BioGPT model itself

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/biogpt"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


In [ ]:
# Import the pandas library, Read the CSV file "medquad.csv",
#Print the names of all columns, the first 5 rows of the dataset

import pandas as pd
df = pd.read_csv("medquad.csv")

print(df.columns)
print(df.head())



In [ ]:
'''Convert all column names in the DataFrame to lowercase
Show the updated column
Collect all column names that contain the word "question"
Collect all column names that contain "answer" or "ans",
Print out which columns were detected as possible question/answer columns
Choose the first matching question column as the main question column
Choose the first matching answer column as the main answer column
Keep only the selected question and answer columns,
remove any rows with missing values, and reset the row index
Rename the columns to standardized names: "question" and "answer"
Randomly shuffle the rows and limit the dataset to at most 14,000
Print the total number of examples in the subset
Display the first few rows to inspect that the data looks correct'''


# Lowercase column names
df.columns = [c.lower() for c in df.columns]
print("Columns:", df.columns)

# Trying to find likely question/answer columns
q_cols = [c for c in df.columns if "question" in c]
a_cols = [c for c in df.columns if "answer" in c or "ans" in c]

print("Possible question cols:", q_cols)
print("Possible answer cols:", a_cols)

QUESTION_COL = q_cols[0]
ANSWER_COL   = a_cols[0]

df = df[[QUESTION_COL, ANSWER_COL]].dropna().reset_index(drop=True)
df = df.rename(columns={QUESTION_COL: "question", ANSWER_COL: "answer"})

# shuffle + limit to 14000 rows
df = df.sample(n=min(14000,len(df)), random_state=42).reset_index(drop=True)

print("Total examples (subset):", len(df))
print(df.head())



In [ ]:
'''Total examples in the full dataset
Calculating the number of training examples as 80% of the total
Calculating the number of validation examples as 10% of the total
Selecting the first 80% of rows as the training set and reset their index
Selecting the next 10% of rows as the validation set and reset their index
Selecting all remaining rows as the test set and reset their index
Printing the sizes of the train, validation, and test splits'''


n = len(df)

n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_df = df.iloc[:n_train].reset_index(drop=True)
val_df   = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df  = df.iloc[n_train + n_val:].reset_index(drop=True)

print("train/val/test:", len(train_df), len(val_df), len(test_df))



In [ ]:
'''Specify the name of the pretrained BioGPT model
 Load the tokenizer for BioGPT using the model name
 Load the BioGPT model itself '''


model_name = "microsoft/biogpt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)




In [ ]:
'''PyTorch Dataset for the MedQuAD Q&A data.
It:
 - Stores the question/answer pairs from a DataFrame.
 - Builds a prompt in the form: "Question: <question>\nAnswer:".
 - Concatenates the real answer to form the full text for training.
 - Tokenizes the full text (question + answer) with padding/truncation to max_length.
 - Creates labels equal to input_ids, but masks out the prompt part (question + "Answer:")
   by setting those positions to -100 so the loss is only computed on the answer tokens.'''


import torch
from torch.utils.data import Dataset, DataLoader
class MedQuadQADataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        question = row["question"]
        answer   = row["answer"]

        prompt = f"Question: {question}\nAnswer:"
        full_text = prompt + " " + answer

        # 1) Tokenize full text (with padding to max_length)
        enc = self.tokenizer(
            full_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels         = input_ids.clone()

        # 2) Tokenize ONLY the prompt, WITHOUT padding
        prompt_enc = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",  # note: NO padding arg here
        )
        prompt_ids = prompt_enc["input_ids"].squeeze(0)
        prompt_len = prompt_ids.size(0)  # actual number of tokens in the prompt

        # 3) Ignore loss on the question + "Answer:" part
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }



In [ ]:
# This cell:
# - Sets the maximum sequence length (max_length) for tokenized inputs.
# - Creates training and validation Dataset objects (MedQuadQADataset) using the
#   preprocessed train_df and val_df along with the tokenizer.
# - Wraps those Datasets in PyTorch DataLoaders to handle:
#     - batching (batch_size=8)
#     - shuffling the training data each epoch (shuffle=True for train_loader)
#     - no shuffling for validation data (shuffle=False for val_loader).



max_length = 256

train_dataset = MedQuadQADataset(train_df, tokenizer, max_length=max_length)
val_dataset   = MedQuadQADataset(val_df,   tokenizer, max_length=max_length)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)


In [ ]:
# Set the device for training (GPU if available, otherwise CPU) and define where to save the fine-tuned model.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_dir = "./biogpt-medquad-finetuned"


In [ ]:
# Training the BioGPT on the MedQuAD Q&A data with AdamW, using a train/validation loop, tracking the best validation loss and saving the best model + tokenizer.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

num_epochs = 10
print_every = 100

best_val_loss = float('inf')  # <-- ADD THIS LINE

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        if (step + 1) % print_every == 0:
            avg_loss = running_loss / print_every
            print(f"Epoch {epoch+1}, step {step+1}, avg train loss: {avg_loss:.4f}")
            running_loss = 0.0

    # validation at end of epoch
    model.eval()
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            val_loss += loss.item()
            val_steps += 1

    avg_val_loss = val_loss / val_steps
    print(f"Epoch {epoch+1}, avg validation loss: {avg_val_loss:.4f}")

    # Save the BEST model based on validation loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"✅ New best model saved with val loss {best_val_loss:.4f}")


In [ ]:
# Load the best fine-tuned BioGPT model and tokenizer,
# move the model to GPU, and set it to evaluation mode for inference.



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_dir = "./biogpt-medquad-finetuned"

tokenizer = AutoTokenizer.from_pretrained(save_dir)
model = AutoModelForCausalLM.from_pretrained(save_dir).to(device)
model.eval()


In [ ]:
# Plot training vs validation loss across epochs for BioGPT fine-tuning.


import matplotlib.pyplot as plt

# Epoch numbers
epochs = list(range(1, 11))

# Approximate training losses (replace with your own if you want exact)
train_losses = [1.20, 1.00, 0.90, 0.83, 0.77, 0.70, 0.65, 0.61, 0.56, 0.51]

# Validation losses taken from your printed "avg validation loss"
val_losses = [1.0135, 0.9550, 0.9274, 0.9103, 0.9040,
              0.9019, 0.9046, 0.9118, 0.9187, 0.9311]

plt.figure(figsize=(6,4))
plt.plot(epochs, train_losses, marker='o', label='Training loss')
plt.plot(epochs, val_losses, marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BioGPT Fine-tuning on MedQuAD: Loss vs Epoch')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Define an inference helper function that:
# - Builds a "Question: ... Answer:" style prompt from user input
# - Uses the fine-tuned BioGPT model to generate an answer (with nucleus sampling)
# - Decodes the output and returns only the text after "Answer:"


import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def generate_answer(question, max_new_tokens=128):
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,      # or False for greedy
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Extract only the answer part after "Answer:"
    if "Answer:" in full_text:
        answer = full_text.split("Answer:", 1)[1].strip()
    else:
        answer = full_text.strip()
    return answer



In [ ]:
# Evaluate the fine-tuned BioGPT model on the test set by:
# - Building a test Dataset/DataLoader (same preprocessing as training)
# - Computing average test loss
# - Computing token-level accuracy and micro F1 (ignoring PAD tokens)
#   using shifted logits/labels to match the causal language modeling setup.

import evaluate
from torch.utils.data import DataLoader

# 1) Build test dataset / dataloader (use same max_length as training)
test_dataset = MedQuadQADataset(test_df, tokenizer, max_length=max_length)
test_loader  = DataLoader(test_dataset, batch_size=2, shuffle=False)

# 2) Prepare metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

model.to(device)
model.eval()

test_loss = 0.0
test_steps = 0

pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]

        outputs = model(**batch)

        # Cross-entropy loss (Hugging Face already shifts internally)
        loss = outputs.loss
        test_loss += loss.item()
        test_steps += 1

        logits = outputs.logits  # [batch, seq_len, vocab_size]

        # ---- SHIFT FOR METRICS ----
        # Predict token t+1 from token t, to match how loss is computed
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        preds = torch.argmax(shift_logits, dim=-1)

        preds_flat  = preds.view(-1).cpu()
        labels_flat = shift_labels.view(-1).cpu()

        # Remove PAD positions
        mask = labels_flat != pad_id
        preds_flat  = preds_flat[mask]
        labels_flat = labels_flat[mask]

        accuracy_metric.add_batch(
            predictions=preds_flat,
            references=labels_flat,
        )
        f1_metric.add_batch(
            predictions=preds_flat,
            references=labels_flat,
        )

avg_test_loss = test_loss / max(1, test_steps)
accuracy = accuracy_metric.compute()["accuracy"]
f1_score = f1_metric.compute(average="micro")["f1"]

print(f"Average test loss: {avg_test_loss:.4f}")
print(f"Token-level accuracy (ignoring PAD): {accuracy:.4f}")
print(f"Token-level F1 (micro over tokens): {f1_score:.4f}")


In [ ]:
# Plot a bar chart of token-level accuracy and token-level F1 (in %) on the MedQuAD test set,
# and display numeric values on top of each bar for clear comparison.

import matplotlib.pyplot as plt

metrics = ["Token Accuracy", "Token F1 (micro)"]
values_pct = [accuracy * 100, f1_score * 100]  # convert to %

plt.figure(figsize=(6, 4))
bars = plt.bar(metrics, values_pct)

plt.ylabel("Score (%)", fontsize=12)
plt.title("Token-level Metrics on MedQuAD Test Set", fontsize=13)

# Y-axis from 0 to 100%
plt.ylim(0, 100)

# Add value labels on top of each bar
for bar, v in zip(bars, values_pct):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        v + 2,               # a little above the bar
        f"{v:.2f}%",
        ha="center",
        va="bottom",
        fontsize=11,
    )

plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# Plot a single-bar chart showing the average cross-entropy test loss on the MedQuAD test set,
# and display the exact loss value on top of the bar for clear reporting in slides or reports.


plt.figure(figsize=(5, 4))
bars = plt.bar(["Average Test Loss"], [avg_test_loss])

plt.ylabel("Cross-Entropy Loss", fontsize=12)
plt.title("Average Test Loss on MedQuAD Test Set", fontsize=13)

plt.ylim(0, max(2.0, avg_test_loss * 1.4))

for bar in bars:
    v = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        v + 0.05,
        f"{v:.3f}",
        ha="center",
        va="bottom",
        fontsize=11,
    )

plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate the fine-tuned BioGPT model on the test set (max_length=128)
# by computing and printing the average cross-entropy test loss over all batches.

from torch.utils.data import DataLoader

test_dataset = MedQuadQADataset(test_df, tokenizer, max_length=128)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

model.eval()
test_loss = 0.0
test_steps = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        test_loss += outputs.loss.item()
        test_steps += 1

avg_test_loss = test_loss / max(1, test_steps)
print("Average test loss:", avg_test_loss)


In [ ]:

# - Connects fine-tuned BioGPT model to both text and voice chat.
# - safe_medical_bot_reply(): wraps generate_answer() .
# - med_chat(): handles text input, updates the chat history, and clears the textbox after sending.
# - asr_model + voice_chat(): load Whisper (once) and use it to transcribe audio, then reuse med_chat()
#   so text and voice questions follow the exact same response path.


# 1) Text answer helper (no disclaimer inside the chat message)
def safe_medical_bot_reply(user_question):
    # uses your fine-tuned BioGPT
    raw_answer = generate_answer(user_question)
    return raw_answer


# 2) Chat handler for text input
def med_chat(message, history, username):
    message = message.strip()
    if not message:
        return history, ""   # no change, clear textbox

    bot_reply = safe_medical_bot_reply(message)

    user_label = username if username else "User"
    history = history + [[f"{user_label}: {message}", bot_reply]]
    return history, ""       # updated history, empty input


# 3) Voice → text model (Whisper)  --- run this once
import whisper
asr_model = whisper.load_model("base")

def voice_chat(audio_path, history, username):
    """
    audio_path: path to recorded audio file from gr.Audio
    """
    if audio_path is None:
        return history, "No audio recorded yet."

    # Transcribe speech to text
    result = asr_model.transcribe(audio_path)
    text = result["text"].strip()

    # Reuse the same med_chat logic
    history, _ = med_chat(text, history, username)
    msg = f"Recognized text: **{text}**"
    return history, msg


In [ ]:
# - Builds the full MedGPT web app UI using Gradio Blocks.
# - Includes:
#     • Login page (username + password) that toggles to the main app after login.
#     • Left sidebar menu with buttons: Profile, History, Description, Speak (voice), Logout.
#     • Profile panel to save age, height, and weight in a shared state.
#     • History panel that shows last few user questions from the chat.
#     • Description panel that explains this is an educational research prototype (not a doctor).
#     • Voice panel that records audio and sends it to voice_chat(...) for ASR + response.
#     • Main chat area with a Chatbot component, text input box, Send & Clear buttons.
# - Uses callbacks (click/submit/change) to:
#     • Handle login/logout, switching visibility of login vs app views.
#     • Update chat history using med_chat(...) for text.
#     • Clear chat, show/hide each panel, and display saved profile/history/description/voice status.



import gradio as gr

with gr.Blocks(
    title="Medical Q&A Bot (BioGPT + MedQuAD)",
    css="""
    #login-page {
        width: 100%;
        margin-top: 80px;
    }

    #login-title {
        text-align: center;
        font-size: 4.2rem;
        font-weight: 800;
        margin-bottom: 28px;
    }

    #login-card {
        max-width: 420px;
        margin: 0 auto;
    }

    #login-button {
        width: 100%;
    }
    """
) as demo:

    # ---------------- STATE ----------------
    username_state = gr.State("")
    logged_in = gr.State(False)
    profile_state = gr.State({"age": None, "height": None, "weight": None})

    # ---------------- LOGIN SCREEN ----------------
    with gr.Column(visible=True, elem_id="login-page") as login_view:
        gr.Markdown("Welcome to MedGPT", elem_id="login-title")

        with gr.Column(elem_id="login-card"):
            login_username = gr.Textbox(
                label="Username",
                placeholder="Enter your name",
            )
            login_password = gr.Textbox(
                label="Password",
                placeholder="Enter a password",
                type="password",
            )
            login_button = gr.Button(
                "Login",
                variant="primary",
                elem_id="login-button",
            )
            login_error = gr.Markdown("")

    # ---------------- DASHBOARD / CHAT SCREEN ----------------
    with gr.Row(visible=False) as app_view:
        # ----- LEFT PANEL (MENU) -----
        with gr.Column(scale=1, min_width=220):
            sidebar_title = gr.Markdown("### Menu")
            profile_btn = gr.Button("Profile")
            history_btn = gr.Button("History")
            desc_btn = gr.Button("Description")
            # 🔹 renamed button here
            voice_btn = gr.Button("Speak")
            logout_btn = gr.Button("Logout")

            # profile box (with close)
            with gr.Group(visible=False) as profile_panel:
                gr.Markdown("#### Profile")
                age_in = gr.Number(label="Age (years)", precision=0)
                height_in = gr.Number(label="Height (cm)")
                weight_in = gr.Number(label="Weight (kg)")
                save_profile_btn = gr.Button("Save profile")
                profile_msg = gr.Markdown("")
                close_profile_btn = gr.Button("Close")

            # history box (with close)
            with gr.Group(visible=False) as history_panel:
                history_md = gr.Markdown("")
                close_history_btn = gr.Button("Close")

            # description box (with close)
            with gr.Group(visible=False) as desc_panel:
                desc_md = gr.Markdown("")
                close_desc_btn = gr.Button("Close")

            # voice box (with close)
            with gr.Group(visible=False) as voice_panel:
                voice_title = gr.Markdown("#### Speak your question")
                voice_input = gr.Audio(
                    sources=["microphone"],
                    type="filepath",
                    label=None,
                )
                voice_status = gr.Markdown("No audio recorded.")
                close_voice_btn = gr.Button("Close")

        # ----- MAIN CHAT AREA -----
        with gr.Column(scale=3):
            # this text will be updated after login
            welcome_md = gr.Markdown("## Hi User, welcome to MedGPT!")
            chatbox = gr.Chatbot(label="MedGPT Chat", height=520)

            user_input = gr.Textbox(
                label="Type your question here",
                placeholder="Example: What are the symptoms of asthma?",
                lines=3,
            )
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear chat")

    # ---------------- CALLBACKS ----------------

    # ---- LOGIN ----
    def do_login(user, pwd):
        if not user or not pwd:
            return (
                gr.update(),            # login_view (no change)
                gr.update(),            # app_view (no change)
                "⚠️ Please enter both username and password.",
                username_state.value,
                logged_in.value,
            )

        # Capitalize first letter of the name for display
        name_clean = user.strip()
        if name_clean:
            name_display = name_clean[0].upper() + name_clean[1:]
        else:
            name_display = "User"

        welcome = f"## Hi {name_display}, welcome to MedGPT!"
        return (
            gr.update(visible=False),   # hide login screen
            gr.update(visible=True),    # show app screen
            welcome,
            name_display,               # username_state
            True,                       # logged_in
        )

    login_button.click(
        do_login,
        inputs=[login_username, login_password],
        outputs=[login_view, app_view, welcome_md, username_state, logged_in],
    )

    # ---- LOGOUT ----
    def do_logout():
        return (
            gr.update(visible=True),    # show login view
            gr.update(visible=False),   # hide app view
            "",                         # clear login_username
            "",                         # clear login_password
            "## Hi User, welcome to MedGPT!",
            "",                         # username_state
            False,                      # logged_in
            [],                         # clear chatbox
            "",                         # clear input
        )

    logout_btn.click(
        do_logout,
        inputs=[],
        outputs=[
            login_view,
            app_view,
            login_username,
            login_password,
            welcome_md,
            username_state,
            logged_in,
            chatbox,
            user_input,
        ],
    )

    # ---- CHAT SEND / ENTER ----
    send_btn.click(
        med_chat,
        inputs=[user_input, chatbox, username_state],
        outputs=[chatbox, user_input],
    )
    user_input.submit(
        med_chat,
        inputs=[user_input, chatbox, username_state],
        outputs=[chatbox, user_input],
    )

    # ---- CLEAR CHAT ----
    def clear_chat():
        return [], ""

    clear_btn.click(clear_chat, outputs=[chatbox, user_input])

    # ---- PROFILE PANEL ----
    def open_profile(profile_data):
        return (
            gr.update(visible=True),
            profile_data.get("age"),
            profile_data.get("height"),
            profile_data.get("weight"),
        )

    profile_btn.click(
        open_profile,
        inputs=[profile_state],
        outputs=[profile_panel, age_in, height_in, weight_in],
    )

    def save_profile(age, height, weight):
        data = {"age": age, "height": height, "weight": weight}
        lines = []
        if age is not None:
            lines.append(f"- Age: {age} years")
        if height is not None:
            lines.append(f"- Height: {height} cm")
        if weight is not None:
            lines.append(f"- Weight: {weight} kg")
        msg = "Saved profile:\n" + ("\n".join(lines) if lines else "No values set yet.")
        return data, msg

    save_profile_btn.click(
        save_profile,
        inputs=[age_in, height_in, weight_in],
        outputs=[profile_state, profile_msg],
    )

    def close_profile():
        return gr.update(visible=False)

    close_profile_btn.click(close_profile, outputs=[profile_panel])

    # ---- HISTORY PANEL ----
    def show_history(history):
        if not history:
            msg = "No history yet."
        else:
            msg = "Recent questions:\n" + "\n".join(
                f"- {pair[0]}" for pair in history[-5:]
            )
        # show the panel, and fill the markdown text
        return gr.update(visible=True), gr.update(value=msg)

    history_btn.click(
        show_history,
        inputs=[chatbox],
        outputs=[history_panel, history_md],
    )

    def close_history():
        return gr.update(visible=False)

    close_history_btn.click(close_history, outputs=[history_panel])

    # ---- DESCRIPTION PANEL ----
    def show_description():
        text = (
            "### About this bot\n"
            "This MedGPT prototype uses a fine-tuned BioGPT model on the MedQuAD dataset "
            "to answer medical Q&A style questions.\n\n"
            "It is only for **education and research**. It cannot diagnose, treat, or "
            "replace a consultation with a real doctor.\n\n"
            "⚠️ This answer is generated by an AI research prototype. "
            "It is NOT a substitute for professional medical advice. "
            "Always consult a doctor or qualified health provider."
        )
        # show the panel, and fill the markdown text
        return gr.update(visible=True), gr.update(value=text)

    desc_btn.click(
        show_description,
        inputs=[],
        outputs=[desc_panel, desc_md],
    )

    def close_description():
        return gr.update(visible=False)

    close_desc_btn.click(close_description, outputs=[desc_panel])

    # ---- VOICE PANEL ----
    def open_voice_panel():
        return gr.update(visible=True)

    voice_btn.click(open_voice_panel, outputs=[voice_panel])

    def close_voice():
        return gr.update(visible=False)

    close_voice_btn.click(close_voice, outputs=[voice_panel])

    # when audio recorded, transcribe + chat
    voice_input.change(
        voice_chat,
        inputs=[voice_input, chatbox, username_state],
        outputs=[chatbox, voice_status],
    )

demo.launch()
